# ARC-AGI-3 Solver — Qwen3.8-27B-FP8

This notebook runs the **TAAF ARC-AGI-3 solver** using a locally mounted **Qwen3.8-27B-FP8** checkpoint through an OpenAI-compatible **vLLM** inference server.

## Model

- **Model:** `Qwen/Qwen3.8-27B-FP8`
- **Format:** Hugging Face / Safetensors
- **Quantization:** FP8
- **Kaggle Model:** `foysalemonshanto/qwen3-8-27b-fp8-repacked-v1`
- **Variation:** `hf-fp8`
- **Version:** `1`
- **Served model ID:** `Qwen/Qwen3.8-27B-FP8`

### Kaggle model path

```text
/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1

In [ ]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

In [ ]:
# Fail-fast GPU assert: metadata machine_shape + --accelerator alone can still
# bind P100; the competition source attachment is the real RTX Pro 6000 gate.
import subprocess as _sp

_gpu = _sp.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("boot gpu:", (_gpu.stdout or "").strip() or (_gpu.stderr or "").strip())
_gpu_name = (_gpu.stdout or "").upper()
assert "RTX" in _gpu_name and "6000" in _gpu_name, (
    f"GPU misbind — expected RTX Pro 6000, got: {_gpu.stdout!r} {_gpu.stderr!r}"
)


In [ ]:
wheelhouse = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if wheelhouse.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-warn-conflicts",
            "--disable-pip-version-check",
            "--find-links",
            str(wheelhouse),
            "arc-agi",
        ]
    )
elif os.getenv("TAAF_KAGGLE_BUNDLE_DIR"):
    print(f"Competition wheelhouse not found at {wheelhouse}; assuming local debug dependencies are installed.")
else:
    raise RuntimeError(f"Competition wheelhouse not found at {wheelhouse}.")

In [ ]:
# Qwen3.8 / Kaggle input configuration
DATASET_SOURCES: list[str] = [
    "jakobbrggen/taaf-kaggle-source-anim-20260807-anim",
    "driessmit1/arc3-vllm-h100-wheelhouse-v3",
]
KERNEL_SOURCES: list[str] = []

# New private Kaggle Model (Version 1).
QWEN_MODEL_OWNER = "foysalemonshanto"
QWEN_MODEL_SLUG = "qwen3-8-27b-fp8-repacked-v1"
QWEN_MODEL_REF = f"{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}"
QWEN_MODEL_VARIATION = "hf-fp8"
QWEN_MODEL_VERSION = "1"
QWEN_SERVED_MODEL_NAME = "Qwen/Qwen3.8-27B-FP8"
QWEN_MODEL_PATH = Path(
    f"/kaggle/input/models/{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}/"
    f"pytorch/{QWEN_MODEL_VARIATION}/{QWEN_MODEL_VERSION}"
)

DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Keep the whole run offline. vLLM/Transformers must use the mounted files only.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path

    # Prefer the attached bundle whose marker actually exists.
    for root in [Path("/kaggle/input/datasets"), Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent

    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(
        json.dumps(data, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Verify the Qwen3.8 Kaggle Model before any expensive setup work starts.
if not QWEN_MODEL_PATH.is_dir():
    raise FileNotFoundError(
        "Qwen3.8 Kaggle Model is not attached.\n"
        f"Expected path:\n{QWEN_MODEL_PATH}\n\n"
        "Attach: Qwen3.8 27B FP8 Repacked → PyTorch → hf-fp8 → Version 1"
    )

_required_qwen_files = [
    "config.json",
    "model.safetensors.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "outside.safetensors",
    "mtp.safetensors",
    "chat_template.jinja",
]
_missing_qwen_files = [
    name for name in _required_qwen_files if not (QWEN_MODEL_PATH / name).is_file()
]
if _missing_qwen_files:
    raise FileNotFoundError(
        "Qwen3.8 mount is incomplete; missing: " + ", ".join(_missing_qwen_files)
    )

_qwen_layer_shards = sorted(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))
_qwen_safetensors = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
if len(_qwen_layer_shards) != 16 or len(_qwen_safetensors) != 18:
    raise RuntimeError(
        "Unexpected Qwen3.8 checkpoint layout: "
        f"{len(_qwen_layer_shards)} layer shards, "
        f"{len(_qwen_safetensors)} safetensors files."
    )

# Tell setup commands and solver code where Kaggle mounted every attached input.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])

for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# The bundled setup resolver asks for owner/slug. Give it a model ref that maps
# directly to the full Kaggle Model version directory.
kaggle_input_paths[QWEN_MODEL_REF] = str(QWEN_MODEL_PATH)

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_QWEN_MODEL_REF": QWEN_MODEL_REF,
    "TAAF_QWEN_MODEL_PATH": str(QWEN_MODEL_PATH),
    "TAAF_QWEN_SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)

print("\n✅ Qwen3.8 input configuration ready")
print(f"Model ref:       {QWEN_MODEL_REF}")
print(f"Physical path:   {QWEN_MODEL_PATH}")
print(f"Served model:    {QWEN_SERVED_MODEL_NAME}")
print(f"Safetensors:     {len(_qwen_safetensors)}")
print(f"Layer shards:    {len(_qwen_layer_shards)}")
print(f"TAAF input map:  {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


In [ ]:
# Audit the attached inputs that matter for this run.
print("=== TAAF bundle ===")
print(BUNDLE_DIR)
print("Exists:", BUNDLE_DIR.exists())

print("\n=== vLLM wheelhouse ===")
_vllm_wheelhouse = Path(
    "/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3"
)
print(_vllm_wheelhouse)
print("Exists:", _vllm_wheelhouse.exists())

print("\n=== Qwen3.8 Kaggle Model ===")
print(QWEN_MODEL_PATH)
print("Exists:", QWEN_MODEL_PATH.exists())
print("Safetensors:", len(list(QWEN_MODEL_PATH.glob("*.safetensors"))))
print(
    "Repacked layer shards:",
    len(list(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))),
)


In [ ]:
import re


def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []

    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env["HF_HUB_OFFLINE"] = "1"
    env["TRANSFORMERS_OFFLINE"] = "1"
    env.update(_load_setup_env())
    return env


def _replace_python_assignment(
    command: str,
    variable_name: str,
    value: str,
) -> tuple[str, int]:
    """Replace a top-level Python string assignment inside the setup here-doc."""
    pattern = rf"(?m)^{re.escape(variable_name)}\s*=\s*(['\"])[^\r\n]*?\1\s*$"
    replacement = f"{variable_name} = {value!r}"
    return re.subn(pattern, replacement, command, count=1)


def _patch_qwen38_setup_commands(commands: list[str]) -> list[str]:
    """
    Preserve the TAAF deployment setup but replace its model identity with the
    Qwen3.8 Kaggle Model. This avoids copying/forking the large bundled setup
    script and keeps the wheelhouse/GPU/vLLM behavior from the source bundle.
    """
    patched: list[str] = []
    replacement_counts = {
        "MODEL_OWNER": 0,
        "MODEL_SLUG": 0,
        "SERVED_MODEL_NAME": 0,
    }

    replacements = {
        "MODEL_OWNER": QWEN_MODEL_OWNER,
        "MODEL_SLUG": QWEN_MODEL_SLUG,
        "SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    }

    for raw_command in commands:
        command = str(raw_command)

        for variable_name, value in replacements.items():
            command, count = _replace_python_assignment(
                command,
                variable_name,
                value,
            )
            replacement_counts[variable_name] += count

        # Make offline behavior explicit in the child process as well.
        if "def vllm_env()" in command:
            command = command.replace(
                "'VLLM_NO_USAGE_STATS': '1',",
                "'VLLM_NO_USAGE_STATS': '1',\n"
                "            'HF_HUB_OFFLINE': '1',\n"
                "            'TRANSFORMERS_OFFLINE': '1',",
                1,
            )

        patched.append(command)

    missing = [
        name for name, count in replacement_counts.items() if count == 0
    ]
    if missing:
        raise RuntimeError(
            "Could not update the bundled TAAF setup for Qwen3.8. "
            "Missing assignment(s): "
            + ", ".join(missing)
            + ". The attached TAAF bundle's setup_commands.json has changed."
        )

    print("taaf.kaggle: Qwen3.8 setup patch =", replacement_counts, flush=True)
    return patched


def _run_shell_commands(filename: str, *, label: str, check: bool) -> None:
    path = BUNDLE_DIR / filename
    if not path.is_file():
        return

    commands = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(commands, list):
        raise RuntimeError(f"{path} must contain a JSON list of shell commands.")

    if filename == "setup_commands.json":
        commands = _patch_qwen38_setup_commands(commands)

    env = _command_env()
    for command in commands:
        print(f"taaf.kaggle: {label} command: {command}", flush=True)
        result = subprocess.run(
            str(command),
            shell=True,
            check=check,
            cwd=WORKING_DIR,
            env=env,
        )
        if not check and result.returncode != 0:
            print(
                f"taaf.kaggle: {label} command exited with {result.returncode}",
                flush=True,
            )

        # Setup commands may export additional runtime settings.
        env.update(_load_setup_env())
        os.environ.update(env)


# Make bundled TAAF repos importable for this notebook and child Python processes.
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text(
        "".join(f"{entry}\n" for entry in source_entries),
        encoding="utf-8",
    )
    print(
        f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)",
        flush=True,
    )

# Run the TAAF deployment setup, patched to use Qwen3.8.
_run_shell_commands("setup_commands.json", label="setup", check=True)

# Setup commands may export PYTHONPATH through TAAF_KAGGLE_SETUP_ENV.
pythonpath_entries = [
    entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry
]
for entry in reversed(pythonpath_entries):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Fail early if the analyzer is still exposing an old model identity.
_actual_model_id = os.environ.get("INFERENCE_ANALYZER_MODEL", "")
if _actual_model_id != QWEN_SERVED_MODEL_NAME:
    raise RuntimeError(
        "TAAF setup completed, but the analyzer model ID is wrong: "
        f"{_actual_model_id!r}; expected {QWEN_SERVED_MODEL_NAME!r}"
    )

print("\n✅ TAAF/vLLM setup completed for Qwen3.8")
print("Model path:", QWEN_MODEL_PATH)
print("Analyzer model:", _actual_model_id)
print("Analyzer endpoint:", os.environ.get("LOCAL_ANALYZER_BASE_URL"))


In [ ]:
# Boot attestation (doctrine v2, 2026-08-17): the mounted weights must be the
# OFFICIAL Qwen3.8-FP8. Discriminators verified offline against both the
# official HF snapshot and the vrfai 3.6 config: 3.8 = quant_method fp8 /
# fmt e4m3 / transformers 5.8.0.dev0; 3.6-vrfai = compressed-tensors
# config_groups / transformers 5.6.2. A wrong mount must DIE here, before any
# game action is spent. --served-model-name is a rename and proves nothing.
import hashlib as _hashlib
import urllib.request as _rq

_cfg_path = QWEN_MODEL_PATH / "config.json"
_cfg_raw = _cfg_path.read_bytes()
_cfg = json.loads(_cfg_raw)
_q = _cfg.get("quantization_config") or {}
assert _cfg.get("architectures") == ["Qwen3_5ForConditionalGeneration"], (
    f"attest FAIL: architectures {_cfg.get('architectures')}")
assert _q.get("quant_method") == "fp8" and _q.get("fmt") == "e4m3", (
    f"attest FAIL: quantization_config is not official fp8/e4m3: {_q}")
assert _cfg.get("transformers_version") == "5.8.0.dev0", (
    f"attest FAIL: transformers_version {_cfg.get('transformers_version')} "
    "(vrfai 3.6 stamps 5.6.2)")
print("attest: config sha256", _hashlib.sha256(_cfg_raw).hexdigest())

_idx_path = QWEN_MODEL_PATH / "model.safetensors.index.json"
if _idx_path.is_file():
    print("attest: index sha256", _hashlib.sha256(_idx_path.read_bytes()).hexdigest())
_shards = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
assert _shards, "attest FAIL: no safetensors shards at model path"
_total = sum(p.stat().st_size for p in _shards)
print(f"attest: {len(_shards)} shards, {_total} bytes total")
assert _total > 25_000_000_000, f"attest FAIL: total shard bytes {_total} too small for 27B FP8"
_h = _hashlib.sha256()
with open(_shards[0], "rb") as _f:
    _h.update(_f.read(1 << 20))
print("attest: first-shard-1MiB sha256", _h.hexdigest())

# Greedy decode fingerprint — logged (not asserted) for cross-run comparison.
_base = (os.environ.get("LOCAL_ANALYZER_BASE_URL") or "http://127.0.0.1:1234/v1").rstrip("/")
if not _base.endswith("/v1"):
    _base += "/v1"
_body = json.dumps({
    "model": QWEN_SERVED_MODEL_NAME,
    "messages": [{"role": "user", "content": "Reply with exactly the sum of 17 and 25, then the word quack."}],
    "temperature": 0.0,
    "max_tokens": 48,
    "chat_template_kwargs": {"enable_thinking": False},
}).encode()
_req = _rq.Request(_base + "/chat/completions", data=_body, headers={
    "Content-Type": "application/json",
    "Authorization": "Bearer " + (os.environ.get("LOCAL_ANALYZER_API_KEY") or "EMPTY"),
})
with _rq.urlopen(_req, timeout=180) as _resp:
    _reply = json.loads(_resp.read())["choices"][0]["message"].get("content") or ""
print("attest: decode fingerprint", repr(_reply)[:160])
print("attest: decode sha256", _hashlib.sha256(_reply.encode()).hexdigest())
print("attest: OK — official Qwen3.8-FP8 signature verified before any game")


In [ ]:
def _soft_end_time(max_runtime_s: float, *, run_as_submission: bool) -> datetime | None:
    if run_as_submission or max_runtime_s <= 0:
        return None
    budget = max(1.0, max_runtime_s)
    buffer = min(SOFT_DEADLINE_BUFFER_S, budget / 2)
    start = datetime.fromtimestamp(NOTEBOOK_START_EPOCH)
    return start + timedelta(seconds=budget - buffer)


def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ.get("ARC_BASE_URL", "http://gateway:8001/"),
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


@contextlib.contextmanager
def _tee_to_file(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", buffering=1)
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = _Tee(original_stdout, log_file)
    sys.stderr = _Tee(original_stderr, log_file)
    try:
        yield
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()


class _Tee:
    def __init__(self, *streams: TextIO) -> None:
        self._streams = streams

    def write(self, data: str) -> int:
        n = 0
        for stream in self._streams:
            n = stream.write(data)
        return n

    def flush(self) -> None:
        for stream in self._streams:
            stream.flush()

    def isatty(self) -> bool:
        return any(getattr(stream, "isatty", lambda: False)() for stream in self._streams)

In [ ]:
true_submission = _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
run_as_submission = _env_bool("TAAF_RUN_AS_SUBMISSION", False) or true_submission
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if run_as_submission else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if run_as_submission else "0"

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = run_as_submission
target.is_competition_rerun = true_submission
soft_end = _soft_end_time(float(getattr(target, "max_runtime_s", 0.0) or 0.0), run_as_submission=run_as_submission)

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# Smoke/eval hook: a NORMAL COMMIT runs the THROUGHPUT A/B SMOKE on the scored
# GPU class. The scored rerun path (KAGGLE_IS_COMPETITION_RERUN) never enters
# this branch.
#
# Two phases, same 25 public games, eval geometry (7,920 s per game,
# concurrency 28 from the serialized solver): "stock" with every graft seam a
# pass-through (TP_ENABLE=0), then "tp" with the Pack-1 flags on.
GAMES_25 = [
    "ar25-0c556536",
    "bp35-0a0ad940",
    "cd82-fb555c5d",
    "cn04-2fe56bfb",
    "dc22-fdcac232",
    "ft09-0d8bbf25",
    "g50t-5849a774",
    "ka59-38d34dbb",
    "lf52-271a04aa",
    "lp85-305b61c3",
    "ls20-9607627b",
    "m0r0-492f87ba",
    "r11l-495a7899",
    "re86-8af5384d",
    "s5i5-18d95033",
    "sb26-7fbdac44",
    "sc25-635fd71a",
    "sk48-d8078629",
    "sp80-589a99af",
    "su15-1944f8ab",
    "tn36-ef4dde99",
    "tr87-cd924810",
    "tu93-0768757b",
    "vc33-5430563c",
    "wa30-ee6fef47"
]
SMOKE_PHASES = [
    ("stock", GAMES_25, 7920, {"TP_ENABLE": "0"}),
    ("tp", GAMES_25, 7920, {"TP_ENABLE": "1"}),
]
TP_PHASE_ERRORS = []
TP_ALL_RUNS = []

if not run_as_submission:
    import arc_agi
    from taaf.game_api import ArcadeSpec, GameAPI

    def _resolve_env_dir():
        candidates = [
            Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"),
            Path("/kaggle/input/arc-prize-2026-arc-agi-3/environment_files"),
        ]
        for cand in candidates:
            if cand.is_dir():
                return str(cand)
        for hit in Path("/kaggle/input").rglob("environment_files"):
            if hit.is_dir():
                return str(hit)
        raise RuntimeError("environment_files dir not found in /kaggle/input")

    _env_dir = _resolve_env_dir()
    _spec = ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=_env_dir)
    bm.games = [GameAPI(env_name=name, arcade_spec=_spec) for name in SMOKE_PHASES[0][1]]
    bm.n_passes = 1
    bm.game_weights = None
    bm.label = "tp-smoke-" + SMOKE_PHASES[0][0]
    bm.solver.max_runtime_s_per_game = float(SMOKE_PHASES[0][2])
    soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=19800)
    print(f"smoke hook: phases={[(p[0], len(p[1]), p[2], p[3]) for p in SMOKE_PHASES]} "
          f"env_dir={_env_dir} concurrency={bm.solver.concurrency} "
          f"per_game_cap={bm.solver.max_runtime_s_per_game}s soft_end={soft_end}")
else:
    print("scored rerun: smoke hook inert — full competition games")

print("Benchmark analyzer model:", os.environ.get("INFERENCE_ANALYZER_MODEL"))


In [ ]:
# Graft install — ONE graft: throughput (Pack 1). Every TP_* flag is set
# EXPLICITLY so the run log states the whole config; the phase loop flips
# TP_ENABLE only. Stale grafts are purged unconditionally.
os.environ["TP_ENABLE"] = "1"
os.environ["TP_TRIM_LOW_WATER"] = "0.5"
os.environ["TP_CONTEXT_WINDOW"] = "24576"
os.environ["TP_YIELD_SECONDS"] = "900"
os.environ["TP_TOOL_STEPS"] = "8"
os.environ["TP_KEEP_NOTES_ON_GAME_OVER"] = "1"
os.environ["TP_BATCH_CAP"] = "10"
for _stale in ("EFFORT_MEDIUM", "EFFORT_DEAD_RETRY", "YIELD_CARRYOVER", "YIELD_SLICE_CAP",
               "EXPLORER", "EXPLORER_V8"):
    os.environ.pop(_stale, None)

_TP_SOURCE = '"""Throughput graft (Pack 1, 2026-08-29) — plumbing-only changes that raise\nactions-per-game of the anim-bundle ToolAgent. No prompt text changes.\n\nEvidence: docs/research-2026-08-29/R4-harness-throughput-audit.md — every\nplay dies on the 7,920 s clock at ~88 actions/game; each call re-prefills a\n16-20k-token prompt (prefix-cache hit 0-20%) because the trimmer drops one\nblock per turn; the 60 s yield is shorter than one call (26% of wall in\naction-less slices); carried notes are wiped on every GAME_OVER; blind\n20-140-action batches burn efficiency and trigger GAME_OVERs.\n\nSeams (anim bundle inference/agent/tool_agent.py, the code that plays at eval):\n- _trim_messages_for_context (:2056) — hysteresis: when over budget, cut to\n  TP_TRIM_LOW_WATER x budget in one go so the cached prefix survives turns.\n- __init__ — _context_budget_tokens / _yield_seconds / _tool_steps are set\n  from module constants read at import; override per instance from\n  TP_CONTEXT_WINDOW / TP_YIELD_SECONDS / TP_TOOL_STEPS.\n- _update_summarized_knowledge_from_step_summary (:1343) — wipes the carried\n  notes on game_over; keep them (TP_KEEP_NOTES_ON_GAME_OVER). Level-up and\n  run-complete wipes are untouched.\n- _run_python_tool (:1693) + _normalize_python_actions (:1614) — per tool\n  call action budget (TP_BATCH_CAP): requests beyond the cap are truncated,\n  and a further action() call raises ValueError inside the sandbox so the\n  model sees a readable error instead of a blind 100-action walk.\n\nFlags (read at call time; TP_ENABLE=0 turns every seam into a pass-through):\n  TP_ENABLE=1  TP_TRIM_LOW_WATER=0.5  TP_CONTEXT_WINDOW=24576\n  TP_YIELD_SECONDS=900  TP_TOOL_STEPS=8  TP_KEEP_NOTES_ON_GAME_OVER=1\n  TP_BATCH_CAP=10\n\nFail-open: graft logic errors fall through to the stock method; the stock\nmethod is never wrapped in try/except, so its own errors propagate as stock.\n"""\nfrom __future__ import annotations\n\nimport os\nimport threading\nfrom typing import Any\n\n_tls = threading.local()\n_STATE = {"installed": False}\n\nDEFAULT_TRIM_LOW_WATER = 0.5\nDEFAULT_CONTEXT_WINDOW = 24576\nDEFAULT_YIELD_SECONDS = 900.0\nDEFAULT_TOOL_STEPS = 8\nDEFAULT_BATCH_CAP = 10\nBATCH_CAP_MESSAGE = (\n    "action batch cap reached: at most {cap} actions per python tool call. "\n    "Observe the results so far, then call the tool again for more actions."\n)\n_OFF = {"0", "false", "no", "off"}\n\n\n# ----------------------------------------------------------------- flags ---\ndef _env(name: str, default: str) -> str:\n    raw = os.environ.get(name)\n    return default if raw is None or not raw.strip() else raw.strip()\n\n\ndef enabled() -> bool:\n    return _env("TP_ENABLE", "1").lower() not in _OFF\n\n\ndef trim_low_water() -> float:\n    """Fraction of the budget to cut down to when over budget; 1.0 = stock."""\n    if not enabled():\n        return 1.0\n    try:\n        value = float(_env("TP_TRIM_LOW_WATER", str(DEFAULT_TRIM_LOW_WATER)))\n    except ValueError:\n        return DEFAULT_TRIM_LOW_WATER\n    return min(1.0, max(0.05, value))\n\n\ndef context_window() -> int:\n    """Context window for the request budget; 0 = leave stock."""\n    if not enabled():\n        return 0\n    try:\n        return max(0, int(_env("TP_CONTEXT_WINDOW", str(DEFAULT_CONTEXT_WINDOW))))\n    except ValueError:\n        return DEFAULT_CONTEXT_WINDOW\n\n\ndef yield_seconds() -> float:\n    """Turn yield in seconds; -1 = leave stock; 0 = disable the yield."""\n    if not enabled():\n        return -1.0\n    try:\n        return float(_env("TP_YIELD_SECONDS", str(DEFAULT_YIELD_SECONDS)))\n    except ValueError:\n        return DEFAULT_YIELD_SECONDS\n\n\ndef tool_steps() -> int:\n    """Calls per turn; -1 = leave stock; 0 = unlimited."""\n    if not enabled():\n        return -1\n    try:\n        return int(_env("TP_TOOL_STEPS", str(DEFAULT_TOOL_STEPS)))\n    except ValueError:\n        return DEFAULT_TOOL_STEPS\n\n\ndef keep_notes_on_game_over() -> bool:\n    if not enabled():\n        return False\n    return _env("TP_KEEP_NOTES_ON_GAME_OVER", "1").lower() not in _OFF\n\n\ndef batch_cap() -> int:\n    """Requested actions per python tool call; 0 = unlimited."""\n    if not enabled():\n        return 0\n    try:\n        return max(0, int(_env("TP_BATCH_CAP", str(DEFAULT_BATCH_CAP))))\n    except ValueError:\n        return DEFAULT_BATCH_CAP\n\n\n# ------------------------------------------------------------- seam: trim ---\ndef _patch_trim(cls: Any) -> None:\n    stock_trim = cls._trim_messages_for_context\n\n    def trim(self, messages, *, tools=None, preserve_recent=1, extra_safety_tokens=0):\n        low = trim_low_water()\n        if low >= 1.0 or not messages:\n            return stock_trim(self, messages, tools=tools, preserve_recent=preserve_recent,\n                              extra_safety_tokens=extra_safety_tokens)\n        try:\n            system_message = messages[0]\n            history = list(messages[1:])\n            preserve_recent = max(0, preserve_recent)\n            budget = max(1, self._context_budget_tokens - max(0, extra_safety_tokens))\n            estimate = self._estimate_request_input_tokens([system_message, *history], tools=tools)\n            if estimate <= budget:\n                return [system_message, *self._drop_until_first_user_message(history)]\n            target = max(1, int(budget * low))\n            while history and estimate > target:\n                if not self._drop_oldest_history_block(history, preserve_recent=preserve_recent):\n                    break\n                estimate = self._estimate_request_input_tokens([system_message, *history], tools=tools)\n            history = self._drop_until_first_user_message(history)\n            return [system_message, *history]\n        except Exception:  # noqa: BLE001 — fail open to stock\n            return stock_trim(self, messages, tools=tools, preserve_recent=preserve_recent,\n                              extra_safety_tokens=extra_safety_tokens)\n\n    trim._tp_stock = stock_trim\n    cls._trim_messages_for_context = trim\n\n\n# ------------------------------------------------------------- seam: init ---\ndef _patch_init(cls: Any) -> None:\n    stock_init = cls.__init__\n\n    def init(self, *args, **kwargs):\n        stock_init(self, *args, **kwargs)\n        try:\n            window = context_window()\n            if window > 0:\n                self._context_budget_tokens = max(\n                    1024, window - self._reply_reserve_tokens - self._request_safety_margin_tokens)\n            ys = yield_seconds()\n            if ys >= 0:\n                self._yield_seconds = None if ys == 0 else float(ys)\n            ts = tool_steps()\n            if ts >= 0:\n                self._tool_steps = None if ts == 0 else max(1, ts)\n        except Exception:  # noqa: BLE001\n            pass\n\n    init._tp_stock = stock_init\n    cls.__init__ = init\n\n\n# ------------------------------------------------------------ seam: notes ---\ndef _patch_notes(cls: Any) -> None:\n    stock = cls._update_summarized_knowledge_from_step_summary\n\n    def update(self):\n        if not keep_notes_on_game_over():\n            return stock(self)\n        try:\n            summary = self._last_step_summary\n            if not summary:\n                return None\n            if summary.get("level_transition") or summary.get("run_complete"):\n                return stock(self)\n            return None  # game_over or nothing: keep every carried note\n        except Exception:  # noqa: BLE001\n            return stock(self)\n\n    update._tp_stock = stock\n    cls._update_summarized_knowledge_from_step_summary = update\n\n\n# -------------------------------------------------------- seam: batch cap ---\ndef begin_tool_call() -> None:\n    """Reset the per-tool-call action budget (called at every _run_python_tool)."""\n    _tls.remaining = batch_cap()\n\n\ndef _remaining() -> int | None:\n    cap = batch_cap()\n    if cap <= 0:\n        return None\n    remaining = getattr(_tls, "remaining", None)\n    if remaining is None:\n        remaining = cap\n        _tls.remaining = remaining\n    return remaining\n\n\ndef _patch_batch_cap(cls: Any) -> None:\n    stock_normalize = cls._normalize_python_actions\n    stock_run = cls._run_python_tool\n\n    def normalize(self, value):\n        normalized = stock_normalize(self, value)\n        try:\n            remaining = _remaining()\n        except Exception:  # noqa: BLE001\n            return normalized\n        if remaining is None:\n            return normalized\n        if remaining <= 0:\n            raise ValueError(BATCH_CAP_MESSAGE.format(cap=batch_cap()))\n        if len(normalized) > remaining:\n            normalized = normalized[:remaining]\n        _tls.remaining = remaining - len(normalized)\n        return normalized\n\n    def run(self, state_path, arguments):\n        begin_tool_call()\n        return stock_run(self, state_path, arguments)\n\n    normalize._tp_stock = stock_normalize\n    run._tp_stock = stock_run\n    cls._normalize_python_actions = normalize\n    cls._run_python_tool = run\n\n\n# ------------------------------------------------------------ time guard ---\ndef time_guard_per_game_s(stock_per_game_s: float, *, setup_elapsed_s: float, games: int,\n                          concurrency: int, total_budget_s: float = 32400.0,\n                          margin_s: float = 240.0) -> float:\n    """Shrink the per-game box only if setup + waves would overrun the 9 h box.\n\n    Never grows the box; never returns less than 600 s.\n    """\n    try:\n        waves = max(1, -(-int(games) // max(1, int(concurrency))))\n        fits = (float(total_budget_s) - float(setup_elapsed_s) - float(margin_s)) / waves\n        return float(min(float(stock_per_game_s), max(600.0, fits)))\n    except Exception:  # noqa: BLE001\n        return float(stock_per_game_s)\n\n\n# --------------------------------------------------------------- install ---\ndef install() -> str:\n    if _STATE["installed"]:\n        return "throughput: SKIP (already applied)"\n    try:\n        from inference.agent import tool_agent as agent_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"throughput: SKIP (tool_agent module missing: {exc!r})"\n    cls = getattr(agent_mod, "ToolAgent", None)\n    if cls is None:\n        return "throughput: SKIP (missing ToolAgent)"\n    for name in ("_trim_messages_for_context", "_update_summarized_knowledge_from_step_summary",\n                 "_normalize_python_actions", "_run_python_tool", "_estimate_request_input_tokens",\n                 "_drop_oldest_history_block", "_drop_until_first_user_message"):\n        if getattr(cls, name, None) is None:\n            return f"throughput: SKIP (ToolAgent.{name} missing)"\n    _patch_trim(cls)\n    _patch_init(cls)\n    _patch_notes(cls)\n    _patch_batch_cap(cls)\n    _STATE["installed"] = True\n    return "throughput: OK"\n\n\ndef status() -> dict[str, Any]:\n    return {\n        "installed": _STATE["installed"],\n        "enabled": enabled(),\n        "trim_low_water": trim_low_water(),\n        "context_window": context_window(),\n        "yield_seconds": yield_seconds(),\n        "tool_steps": tool_steps(),\n        "keep_notes_on_game_over": keep_notes_on_game_over(),\n        "batch_cap": batch_cap(),\n    }\n'
_TP_DIR = WORKING_DIR / "tp_bundle"
_TP_DIR.mkdir(parents=True, exist_ok=True)
(_TP_DIR / "graft_throughput.py").write_text(_TP_SOURCE, encoding="utf-8")
if str(_TP_DIR) not in sys.path:
    sys.path.insert(0, str(_TP_DIR))

import importlib as _importlib

_tpmod = _importlib.import_module("graft_throughput")
_tp_status = _tpmod.install()
print("[throughput]", _tp_status)
assert _tp_status == "throughput: OK", "throughput graft must be live, got: " + repr(_tp_status)
print("[throughput] flags:", {k: v for k, v in os.environ.items() if k.startswith("TP_")})
print("[throughput] status:", _tpmod.status())


In [ ]:
# ---- tp smoke telemetry: THE READ. Per phase: per-game actions / levels /
# score / turns / tokens from the framework mirror + the ToolAgent session
# counters, and vLLM /metrics deltas (prompt + generation tokens, prefix-cache
# queries/hits, request count) — the mechanical quantities Pack 1 targets.
import json as _tel_json
import re as _tel_re
import threading as _tel_threading
import time as _tel_time
import urllib.request as _tel_rq

from inference.framework import solver as _solver_mod

_tel_lock = _tel_threading.Lock()
_TEL_PATH = WORKING_DIR / "tp_smoke_results.json"
TP_SESSIONS = {}      # phase -> {game_id: {...}} filled at session exit
TP_PHASES = []        # ordered phase records
_TP_CURRENT = {"phase": None, "t0": None, "metrics0": None}


def _metrics_url():
    base = (os.environ.get("LOCAL_ANALYZER_BASE_URL") or "http://127.0.0.1:1234/v1").rstrip("/")
    if base.endswith("/v1"):
        base = base[:-3]
    return base + "/metrics"


_METRIC_KEYS = (
    "vllm:prompt_tokens_total", "vllm:generation_tokens_total",
    "vllm:prefix_cache_queries_total", "vllm:prefix_cache_hits_total",
    "vllm:request_success_total", "vllm:num_preemptions_total",
)


def _scrape_metrics():
    out = {}
    try:
        with _tel_rq.urlopen(_metrics_url(), timeout=20) as resp:
            text = resp.read().decode("utf-8", errors="replace")
    except Exception as exc:  # noqa: BLE001
        return {"error": repr(exc)}
    for line in text.splitlines():
        if not line or line.startswith("#"):
            continue
        for key in _METRIC_KEYS:
            if line.startswith(key):
                m = _tel_re.match(r"^\S+(?:\{[^}]*\})?\s+([0-9.eE+-]+)", line)
                if m:
                    try:
                        out[key] = out.get(key, 0.0) + float(m.group(1))
                    except ValueError:
                        pass
    return out


def _tp_phase_begin(phase):
    with _tel_lock:
        _TP_CURRENT["phase"] = phase
        _TP_CURRENT["t0"] = _tel_time.monotonic()
        _TP_CURRENT["metrics0"] = _scrape_metrics()
        TP_SESSIONS.setdefault(phase, {})
    print(f"[tp-tel] phase {phase} begin metrics0={_TP_CURRENT['metrics0']}", flush=True)


def _tp_phase_end(phase, game_runs):
    m1 = _scrape_metrics()
    m0 = _TP_CURRENT.get("metrics0") or {}
    delta = {k: (m1.get(k, 0.0) - m0.get(k, 0.0)) for k in _METRIC_KEYS if k in m1}
    wall = _tel_time.monotonic() - (_TP_CURRENT.get("t0") or _tel_time.monotonic())
    games = []
    for game_run in game_runs:
        apl = list(game_run.actions_per_level or [])
        actions = sum(apl) if apl else len(game_run.history)
        sess = TP_SESSIONS.get(phase, {}).get(game_run.game_id, {})
        games.append({
            "game_id": game_run.game_id,
            "state": game_run.state,
            "levels_completed": game_run.levels_completed,
            "number_of_levels": game_run.number_of_levels,
            "final_score": game_run.final_score,
            "actions": actions,
            "actions_per_level": apl,
            "base_actions_per_level": list(game_run.base_actions_per_level or []),
            "wallclock_s": game_run.final_wallclock_seconds,
            "solver_note": game_run.solver_note,
            "turns": sess.get("turns"),
            "session_total_tokens": sess.get("total_tokens"),
            "session_generated_tokens": sess.get("generated_tokens"),
            "history_messages_at_exit": sess.get("history_messages"),
            "context_budget_tokens": sess.get("context_budget_tokens"),
            "yield_seconds": sess.get("yield_seconds"),
            "tool_steps": sess.get("tool_steps"),
        })
    n = max(1, len(games))
    queries = delta.get("vllm:prefix_cache_queries_total", 0.0)
    hits = delta.get("vllm:prefix_cache_hits_total", 0.0)
    gen = delta.get("vllm:generation_tokens_total", 0.0)
    prompt = delta.get("vllm:prompt_tokens_total", 0.0)
    rec = {
        "phase": phase,
        "env": {k: v for k, v in os.environ.items() if k.startswith("TP_")},
        "graft_status": _tpmod.status(),
        "wall_s": round(wall, 1),
        "games": games,
        "n_games": len(games),
        "mean_actions": round(sum(g["actions"] for g in games) / n, 2),
        "mean_levels": round(sum(g["levels_completed"] for g in games) / n, 3),
        "mean_score": round(sum(float(g["final_score"] or 0.0) for g in games) / n, 4),
        "zero_level_games": sum(1 for g in games if g["levels_completed"] == 0),
        "mean_turns": round(sum(float(g["turns"] or 0) for g in games) / n, 1),
        "metrics_delta": delta,
        "prefix_hit_rate": round(hits / queries, 4) if queries else None,
        "prefill_per_gen_token": round(prompt / gen, 2) if gen else None,
        "gen_tok_s": round(gen / wall, 1) if wall else None,
    }
    with _tel_lock:
        TP_PHASES.append(rec)
    try:
        _TEL_PATH.write_text(_tel_json.dumps(
            {"phases": TP_PHASES, "phase_errors": TP_PHASE_ERRORS}, indent=1, default=str),
            encoding="utf-8")
    except Exception:  # noqa: BLE001
        pass
    print(f"[tp-tel] phase {phase} end: games={rec['n_games']} mean_actions={rec['mean_actions']} "
          f"mean_levels={rec['mean_levels']} mean_score={rec['mean_score']} "
          f"zero_level={rec['zero_level_games']} turns={rec['mean_turns']} "
          f"prefix_hit={rec['prefix_hit_rate']} prefill/gen={rec['prefill_per_gen_token']} "
          f"gen_tok_s={rec['gen_tok_s']} wall={rec['wall_s']}s", flush=True)


# Session exit seam: record turns + token counters per game for the phase.
_inner_play = _solver_mod._HarnessGameSession.play


def _tel_play(self):
    try:
        return _inner_play(self)
    finally:
        try:
            gid = getattr(getattr(self.game, "game_run", None), "game_id", "?")
            an = self.analyzer
            rec = {
                "turns": int(getattr(self, "analysis_step", 0) or 0),
                "total_tokens": int(getattr(an, "_session_total_tokens", 0) or 0),
                "generated_tokens": int(getattr(an, "_session_generated_tokens", 0) or 0),
                "history_messages": len(getattr(an, "_history_messages", []) or []),
                "context_budget_tokens": getattr(an, "_context_budget_tokens", None),
                "yield_seconds": getattr(an, "_yield_seconds", None),
                "tool_steps": getattr(an, "_tool_steps", None),
            }
            with _tel_lock:
                TP_SESSIONS.setdefault(_TP_CURRENT.get("phase") or "?", {})[gid] = rec
        except Exception:  # noqa: BLE001
            pass


if not getattr(_solver_mod._HarnessGameSession.play, "_tp_tel", False):
    _tel_play._tp_tel = True
    _solver_mod._HarnessGameSession.play = _tel_play

print("[tp-tel] installed; metrics url =", _metrics_url(), flush=True)


In [ ]:
run_context = contextlib.nullcontext() if run_as_submission else _tee_to_file(WORKING_DIR / "stdout.log")
with run_context:
    preamble = (BUNDLE_DIR / "preamble.txt").read_text(encoding="utf-8")
    print(preamble)
    print(f"deploy.kaggle: working_dir             = {WORKING_DIR}")
    print(f"deploy.kaggle: run_as_submission       = {run_as_submission}")
    print(f"deploy.kaggle: competition_rerun       = {true_submission}")
    print(f"deploy.kaggle: soft_end_time           = {soft_end}")
    print("---")

    bundled_git_status = BUNDLE_DIR / "git_status.txt"
    if bundled_git_status.is_file():
        (WORKING_DIR / "git_status.txt").write_text(
            bundled_git_status.read_text(encoding="utf-8"),
            encoding="utf-8",
        )

    if true_submission:
        # Competition reruns use Kaggle's live gateway instead of the bundled offline games.
        os.environ.setdefault("ARC_API_KEY", "test-key-123")
        os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
        os.environ.setdefault("SCHEME", "http")
        os.environ.setdefault("HOST", "gateway")
        os.environ.setdefault("PORT", "8001")
        os.environ.setdefault("OPERATION_MODE", "competition")
        os.environ.setdefault("ENVIRONMENTS_DIR", "")
        os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

        deadline = time.monotonic() + 600.0
        last_error = ""
        while time.monotonic() < deadline:
            try:
                with urlopen("http://gateway:8001/api/games", timeout=10) as response:
                    if response.status < 500:
                        break
            except Exception as exc:
                last_error = repr(exc)
            time.sleep(5)
        else:
            raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")

        bm.games = _competition_games()
        bm.n_passes = 1
        bm.game_weights = None

    try:
        # tp smoke: TWO phases (stock, then graft) inside ONE vLLM boot. The
        # scored-rerun path takes exactly one pass with the competition games.
        for _phase_name, _phase_games, _phase_cap, _phase_env in SMOKE_PHASES:
            if not run_as_submission:
                TP_ALL_RUNS.extend(bm.game_runs)
                bm.game_runs = []
                bm.games = [GameAPI(env_name=_n, arcade_spec=_spec) for _n in _phase_games]
                bm.n_passes = 1
                bm.game_weights = None
                bm.label = "tp-smoke-" + _phase_name
                bm.solver.max_runtime_s_per_game = float(_phase_cap)
                for _k, _v in _phase_env.items():
                    os.environ[_k] = _v
                print(f"=== PHASE {_phase_name}: {len(_phase_games)} games "
                      f"per_game_cap={_phase_cap}s env={_phase_env} "
                      f"graft_status={_tpmod.status()} ===", flush=True)
            _tp_phase_begin(_phase_name)
            try:
                await bm.run(
                    soft_end_time=soft_end,
                    runtime_environment=target,
                    minimal_diagnostics=run_as_submission,
                )
            except Exception as _phase_exc:  # noqa: BLE001
                import traceback

                TP_PHASE_ERRORS.append(f"{_phase_name}: {type(_phase_exc).__name__}: {_phase_exc}")
                print(f"PHASE {_phase_name} RAISED {type(_phase_exc).__name__}: {_phase_exc}",
                      flush=True)
                traceback.print_exc()
            try:
                _tp_phase_end(_phase_name, list(bm.game_runs))
            except Exception:  # noqa: BLE001
                pass
            if run_as_submission:
                break
        if not true_submission and Path("/kaggle/input").exists():
            try:
                import pandas as pd

                submission = pd.DataFrame(
                    data=[["1_0", "1", True, 1]],
                    columns=["row_id", "game_id", "end_of_game", "score"],
                )
                submission.to_parquet(WORKING_DIR / "submission.parquet", index=False)
            except Exception as exc:
                print(f"taaf.kaggle: could not write offline dummy submission: {exc!r}", flush=True)
    finally:
        _run_shell_commands("teardown_commands.json", label="teardown", check=False)

In [ ]:
# ---- tp smoke final report (grep for TP SMOKE / PHASE / READ) ----
print("=" * 78)
print("TP SMOKE RESULTS (Pack 1 throughput graft, stock vs tp)")
print("install verdict:", _tp_status)
by = {p["phase"]: p for p in TP_PHASES}
for name in ("stock", "tp"):
    p = by.get(name)
    if not p:
        print(f"PHASE {name}: MISSING")
        continue
    print(f"PHASE {name}: games={p['n_games']} mean_actions={p['mean_actions']} "
          f"mean_levels={p['mean_levels']} mean_score={p['mean_score']} "
          f"zero_level={p['zero_level_games']} mean_turns={p['mean_turns']} "
          f"prefix_hit={p['prefix_hit_rate']} prefill/gen={p['prefill_per_gen_token']} "
          f"gen_tok_s={p['gen_tok_s']} wall={p['wall_s']}s")
    for g in sorted(p["games"], key=lambda g: g["game_id"]):
        print(f"  {g['game_id']}: levels={g['levels_completed']}/{g['number_of_levels']} "
              f"score={g['final_score']} actions={g['actions']} turns={g['turns']} "
              f"gen_tok={g['session_generated_tokens']} state={g['state']}")

verdict = "UNREADABLE"
detail = ""
if "stock" in by and "tp" in by and by["stock"]["n_games"] and by["tp"]["n_games"]:
    s, t = by["stock"], by["tp"]
    ratio = (t["mean_actions"] / s["mean_actions"]) if s["mean_actions"] else float("inf")
    dlev = t["mean_levels"] - s["mean_levels"]
    hit = t["prefix_hit_rate"] or 0.0
    detail = f"actions x{ratio:.2f} levels {dlev:+.3f} prefix_hit {hit:.2f}"
    if ratio >= 2.0 and dlev >= -0.15 and hit >= 0.5:
        verdict = "PASS"
    elif ratio >= 1.5 and dlev >= -0.15:
        verdict = "INCONCLUSIVE"
    else:
        verdict = "FAIL"
print(f"TP SMOKE READ: {verdict} ({detail}) phase_errors={TP_PHASE_ERRORS}")
results = {
    "arm": "tp-smoke",
    "install_verdict": _tp_status,
    "phases": TP_PHASES,
    "phase_errors": TP_PHASE_ERRORS,
    "verdict": verdict,
    "detail": detail,
}
(WORKING_DIR / "tp_smoke_results.json").write_text(
    json.dumps(results, indent=1, default=str), encoding="utf-8")
print("wrote", WORKING_DIR / "tp_smoke_results.json")
